# Inverse-Time Overcurrent Relay Curves

This notebook generates professional inverse-time overcurrent relay characteristic curves for five ANSI-style curve families:

- **U1 — Moderately Inverse**
- **U2 — Inverse**
- **U3 — Very Inverse**
- **U4 — Extremely Inverse**
- **U5 — Short-Time Inverse**

It calculates operating-time and reset-time characteristics, plots multiple Time Dial settings, compares the curve families, validates the equations numerically, and exports the results to CSV, PNG, SVG, and ZIP formats.

The notebook is designed to run directly in **Google Colab** using **Runtime → Run all**.


## Mathematical background

Define

\[
M=\frac{I}{I_{\mathrm{pickup}}}
\]

where \(M\) is the current in multiples of pickup. The Time Dial \(TD\) scales the characteristic vertically.

For operating-time curves, the physically useful region is \(M>1\). As \(M\rightarrow 1^+\), the denominator approaches zero and the operating time grows rapidly.

For reset-time curves, the equations used here are evaluated only in the positive-time region \(0<M<1\). The point \(M=1\) is excluded because it is singular.


## Curve equations

### U1 — Moderately Inverse

\[
t_p = TD\left(0.0226+\frac{0.0104}{M^{0.02}-1}\right)
\]

\[
t_r = TD\left(\frac{1.08}{1-M^2}\right)
\]

### U2 — Inverse

\[
t_p = TD\left(0.180+\frac{5.95}{M^2-1}\right)
\]

\[
t_r = TD\left(\frac{5.95}{1-M^2}\right)
\]

### U3 — Very Inverse

\[
t_p = TD\left(0.0963+\frac{3.88}{M^2-1}\right)
\]

\[
t_r = TD\left(\frac{3.88}{1-M^2}\right)
\]

### U4 — Extremely Inverse

\[
t_p = TD\left(0.0352+\frac{5.67}{M^2-1}\right)
\]

\[
t_r = TD\left(\frac{5.67}{1-M^2}\right)
\]

### U5 — Short-Time Inverse

\[
t_p = TD\left(0.00262+\frac{0.00342}{M^{0.02}-1}\right)
\]

\[
t_r = TD\left(\frac{0.323}{1-M^2}\right)
\]


## Python imports

In [ ]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, NullFormatter

pd.set_option("display.precision", 6)


## User configuration

In [ ]:
# Main user-editable settings
TIME_DIALS = [0.5, 1, 2, 3, 4, 5]
COMPARISON_TD = 1.0
RESET_COMPARISON_TD = 1.0

OPERATING_M_MIN = 1.01
OPERATING_M_MAX = 100.0
RESET_M_MIN = 0.05
RESET_M_MAX = 0.99
N_POINTS = 1200

PICKUP_REFERENCE_M = 1.5

# Use /content in Google Colab and a local folder elsewhere.
BASE_OUTPUT_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
OUTPUT_DIR = BASE_OUTPUT_DIR / "relay_curve_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")


## Relay curve database

In [ ]:
CURVES = {
    "U1": {
        "name": "Moderately Inverse",
        "operating_offset": 0.0226,
        "operating_numerator": 0.0104,
        "operating_exponent": 0.02,
        "reset_numerator": 1.08,
        "slug": "moderately_inverse",
    },
    "U2": {
        "name": "Inverse",
        "operating_offset": 0.180,
        "operating_numerator": 5.95,
        "operating_exponent": 2.0,
        "reset_numerator": 5.95,
        "slug": "inverse",
    },
    "U3": {
        "name": "Very Inverse",
        "operating_offset": 0.0963,
        "operating_numerator": 3.88,
        "operating_exponent": 2.0,
        "reset_numerator": 3.88,
        "slug": "very_inverse",
    },
    "U4": {
        "name": "Extremely Inverse",
        "operating_offset": 0.0352,
        "operating_numerator": 5.67,
        "operating_exponent": 2.0,
        "reset_numerator": 5.67,
        "slug": "extremely_inverse",
    },
    "U5": {
        "name": "Short-Time Inverse",
        "operating_offset": 0.00262,
        "operating_numerator": 0.00342,
        "operating_exponent": 0.02,
        "reset_numerator": 0.323,
        "slug": "short_time_inverse",
    },
}

CURVE_CODES = list(CURVES.keys())
pd.DataFrame.from_dict(CURVES, orient="index")


## Core calculation functions

The calculation functions are separated from the visualization functions so that the same equations can later be reused in relay coordination studies, automated reports, or optimization workflows.


In [ ]:
def _validate_curve_type(curve_type):
    """Return the normalized curve code and validate that it is supported."""
    code = str(curve_type).upper().strip()
    if code not in CURVES:
        raise ValueError(f"Unsupported curve type: {curve_type}. Use one of {CURVE_CODES}.")
    return code


def _return_scalar_if_scalar_input(input_value, result):
    """Return a Python float when the original input was scalar."""
    if np.isscalar(input_value):
        return float(np.asarray(result))
    return result


def relay_operating_time(curve_type, M, TD):
    """Calculate relay operating time in seconds.

    Parameters
    ----------
    curve_type : str
        Curve code: U1, U2, U3, U4, or U5.
    M : float or array-like
        Current in multiples of pickup. Valid operating region: M > 1.
    TD : float
        Time Dial setting.

    Returns
    -------
    float or numpy.ndarray
        Operating time in seconds. Invalid points return NaN.
    """
    code = _validate_curve_type(curve_type)
    params = CURVES[code]

    m = np.asarray(M, dtype=float)
    result = np.full_like(m, np.nan, dtype=float)
    valid = np.isfinite(m) & (m > 1.0) & np.isfinite(TD) & (TD > 0)

    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        denominator = np.power(m[valid], params["operating_exponent"]) - 1.0
        result[valid] = TD * (
            params["operating_offset"]
            + params["operating_numerator"] / denominator
        )

    result[~np.isfinite(result)] = np.nan
    result[result <= 0] = np.nan
    return _return_scalar_if_scalar_input(M, result)


def relay_reset_time(curve_type, M, TD):
    """Calculate relay reset time in seconds.

    Parameters
    ----------
    curve_type : str
        Curve code: U1, U2, U3, U4, or U5.
    M : float or array-like
        Current in multiples of pickup. Valid reset region: 0 < M < 1.
    TD : float
        Time Dial setting.

    Returns
    -------
    float or numpy.ndarray
        Reset time in seconds. Invalid points return NaN.
    """
    code = _validate_curve_type(curve_type)
    params = CURVES[code]

    m = np.asarray(M, dtype=float)
    result = np.full_like(m, np.nan, dtype=float)
    valid = np.isfinite(m) & (m > 0.0) & (m < 1.0) & np.isfinite(TD) & (TD > 0)

    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        denominator = 1.0 - np.power(m[valid], 2.0)
        result[valid] = TD * params["reset_numerator"] / denominator

    result[~np.isfinite(result)] = np.nan
    result[result <= 0] = np.nan
    return _return_scalar_if_scalar_input(M, result)


def generate_operating_curve(
    curve_type,
    TD,
    m_min=OPERATING_M_MIN,
    m_max=OPERATING_M_MAX,
    n_points=N_POINTS,
):
    """Generate logarithmically spaced operating-curve data."""
    m = np.logspace(np.log10(m_min), np.log10(m_max), n_points)
    t = relay_operating_time(curve_type, m, TD)
    return m, t


def generate_reset_curve(
    curve_type,
    TD,
    m_min=RESET_M_MIN,
    m_max=RESET_M_MAX,
    n_points=N_POINTS,
):
    """Generate reset-curve data over the valid M < 1 region."""
    m = np.linspace(m_min, m_max, n_points)
    t = relay_reset_time(curve_type, m, TD)
    return m, t


def build_operating_dataframe(curve_type, time_dials=TIME_DIALS):
    """Build an operating-time DataFrame for one curve family."""
    m, _ = generate_operating_curve(curve_type, TD=1.0)
    data = {"M": m}
    for td in time_dials:
        data[f"TD_{td:g}"] = relay_operating_time(curve_type, m, td)
    return pd.DataFrame(data)


def build_comparison_dataframe(TD=COMPARISON_TD):
    """Build a DataFrame comparing all curve families at one Time Dial."""
    m, _ = generate_operating_curve("U1", TD)
    data = {"M": m}
    for code in CURVE_CODES:
        data[code] = relay_operating_time(code, m, TD)
    return pd.DataFrame(data)


def build_reset_comparison_dataframe(TD=RESET_COMPARISON_TD):
    """Build a reset-time comparison DataFrame."""
    m, _ = generate_reset_curve("U1", TD)
    data = {"M": m}
    for code in CURVE_CODES:
        data[code] = relay_reset_time(code, m, TD)
    return pd.DataFrame(data)


## Plotting utilities

In [ ]:
def configure_log_grid(ax, x_limits=(1, 100), y_limits=(0.01, 100)):
    """Configure an engineering-style log-log plot."""
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)

    ax.xaxis.set_major_locator(LogLocator(base=10.0))
    ax.xaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1))
    ax.yaxis.set_major_locator(LogLocator(base=10.0))
    ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.yaxis.set_minor_formatter(NullFormatter())

    ax.grid(which="major", linewidth=1.0, alpha=0.70)
    ax.grid(which="minor", linewidth=0.55, alpha=0.38)


def save_figure(fig, stem):
    """Save one figure as high-resolution PNG and vector SVG."""
    png_path = OUTPUT_DIR / f"{stem}.png"
    svg_path = OUTPUT_DIR / f"{stem}.svg"
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(svg_path, bbox_inches="tight")
    return png_path, svg_path


def plot_curve_family(
    curve_type,
    time_dials=TIME_DIALS,
    pickup_reference=PICKUP_REFERENCE_M,
    save=True,
):
    """Plot all configured Time Dial curves for one relay curve family."""
    code = _validate_curve_type(curve_type)
    params = CURVES[code]

    fig, ax = plt.subplots(figsize=(10, 8))

    line_records = []
    for td in time_dials:
        m, t = generate_operating_curve(code, td)
        line, = ax.plot(m, t, linewidth=2.4, label=f"TD = {td:g}")
        line_records.append((td, line, m, t))

    configure_log_grid(ax)
    ax.set_xlabel("CURRENT IN MULTIPLES OF PICKUP", fontsize=12, fontweight="bold")
    ax.set_ylabel("TIME IN SECONDS", fontsize=12, fontweight="bold")
    ax.set_title(f"{code} — {params['name'].upper()}", fontsize=16, fontweight="bold", pad=14)

    ax.axvline(pickup_reference, linestyle="--", linewidth=1.5)
    ax.text(
        pickup_reference * 0.97,
        0.015,
        "1.5 × MULTIPLE OF PICKUP",
        rotation=90,
        va="bottom",
        ha="right",
        fontsize=9,
    )

    # Direct labels near the right side of the curves, similar to relay manuals.
    label_m = 30.0
    ax.text(label_m * 1.02, 0.95, "TIME\nDIAL", fontsize=10, fontweight="bold")
    for td, line, _, _ in line_records:
        y = relay_operating_time(code, label_m, td)
        if np.isfinite(y) and 0.01 <= y <= 100:
            ax.text(
                label_m * 1.08,
                y,
                f"{td:g}",
                color=line.get_color(),
                fontsize=10,
                va="center",
                fontweight="bold",
            )

    ax.legend(loc="upper right", frameon=True, title="Time Dial")
    fig.tight_layout()

    if save:
        save_figure(fig, f"{code}_{params['slug']}")
    return fig, ax


def plot_curve_comparison(TD=COMPARISON_TD, save=True):
    """Compare all operating-time curve families at one Time Dial."""
    fig, ax = plt.subplots(figsize=(10, 8))

    for code in CURVE_CODES:
        m, t = generate_operating_curve(code, TD)
        ax.plot(m, t, linewidth=2.4, label=f"{code} — {CURVES[code]['name']}")

    configure_log_grid(ax)
    ax.set_xlabel("CURRENT IN MULTIPLES OF PICKUP", fontsize=12, fontweight="bold")
    ax.set_ylabel("TIME IN SECONDS", fontsize=12, fontweight="bold")
    ax.set_title("Comparison of Inverse-Time Relay Curves", fontsize=16, fontweight="bold", pad=14)
    ax.text(0.02, 0.95, f"TD = {TD:g}", transform=ax.transAxes, fontsize=11, fontweight="bold")

    ax.axvline(PICKUP_REFERENCE_M, linestyle="--", linewidth=1.5)
    ax.legend(loc="upper right", frameon=True)
    fig.tight_layout()

    if save:
        save_figure(fig, f"comparison_TD{TD:g}")
    return fig, ax


def plot_reset_comparison(TD=RESET_COMPARISON_TD, save=True):
    """Compare reset-time characteristics in the valid M < 1 region."""
    fig, ax = plt.subplots(figsize=(10, 7))

    for code in CURVE_CODES:
        m, t = generate_reset_curve(code, TD)
        ax.plot(m, t, linewidth=2.4, label=f"{code} — {CURVES[code]['name']}")

    ax.set_yscale("log")
    ax.set_xlim(RESET_M_MIN, 1.0)
    ax.set_xlabel("CURRENT / PICKUP CURRENT, M", fontsize=12, fontweight="bold")
    ax.set_ylabel("RESET TIME IN SECONDS", fontsize=12, fontweight="bold")
    ax.set_title("Reset-Time Curve Comparison", fontsize=16, fontweight="bold", pad=14)
    ax.grid(which="major", linewidth=1.0, alpha=0.70)
    ax.grid(which="minor", linewidth=0.55, alpha=0.38)
    ax.legend(loc="upper left", frameon=True)
    ax.text(0.02, 0.94, f"TD = {TD:g}", transform=ax.transAxes, fontsize=11, fontweight="bold")
    fig.tight_layout()

    if save:
        save_figure(fig, f"reset_comparison_TD{TD:g}")
    return fig, ax


## U1 — Moderately Inverse

Increasing the Time Dial shifts the characteristic upward because operating time is directly proportional to \(TD\).


In [ ]:
df_u1 = build_operating_dataframe("U1")
display(df_u1.head())
fig_u1, ax_u1 = plot_curve_family("U1")
plt.show()


## U2 — Inverse

In [ ]:
df_u2 = build_operating_dataframe("U2")
display(df_u2.head())
fig_u2, ax_u2 = plot_curve_family("U2")
plt.show()


## U3 — Very Inverse

In [ ]:
df_u3 = build_operating_dataframe("U3")
display(df_u3.head())
fig_u3, ax_u3 = plot_curve_family("U3")
plt.show()


## U4 — Extremely Inverse

In [ ]:
df_u4 = build_operating_dataframe("U4")
display(df_u4.head())
fig_u4, ax_u4 = plot_curve_family("U4")
plt.show()


## U5 — Short-Time Inverse

In [ ]:
df_u5 = build_operating_dataframe("U5")
display(df_u5.head())
fig_u5, ax_u5 = plot_curve_family("U5")
plt.show()


## Comparison of all curve types

This comparison uses the same Time Dial for all five families. The different curve shapes illustrate how strongly operating time changes with current multiple for each characteristic.


In [ ]:
comparison_df = build_comparison_dataframe(COMPARISON_TD)
display(comparison_df.head())

fig_comparison, ax_comparison = plot_curve_comparison(COMPARISON_TD)
plt.show()


## Reset-time curves

Reset curves use the region \(0<M<1\). The singular point at \(M=1\) is intentionally excluded.


In [ ]:
reset_comparison_df = build_reset_comparison_dataframe(RESET_COMPARISON_TD)
display(reset_comparison_df.head())

fig_reset, ax_reset = plot_reset_comparison(RESET_COMPARISON_TD)
plt.show()


## Engineering example

In [ ]:
pickup_current_a = 100.0
actual_current_a = 500.0
example_td = 1.0

M_example = actual_current_a / pickup_current_a

example_rows = []
for code in CURVE_CODES:
    example_rows.append(
        {
            "Curve": code,
            "Description": CURVES[code]["name"],
            "Pickup Current [A]": pickup_current_a,
            "Actual Current [A]": actual_current_a,
            "M": M_example,
            "Time Dial": example_td,
            "Operating Time [s]": relay_operating_time(code, M_example, example_td),
        }
    )

example_df = pd.DataFrame(example_rows)
display(example_df)


## Export numerical data

In [ ]:
curve_dataframes = {
    "U1": df_u1,
    "U2": df_u2,
    "U3": df_u3,
    "U4": df_u4,
    "U5": df_u5,
}

for code, dataframe in curve_dataframes.items():
    params = CURVES[code]
    csv_path = OUTPUT_DIR / f"{code}_{params['slug']}.csv"
    dataframe.to_csv(csv_path, index=False)

comparison_csv = OUTPUT_DIR / f"curve_comparison_TD{COMPARISON_TD:g}.csv"
reset_csv = OUTPUT_DIR / f"reset_curve_comparison_TD{RESET_COMPARISON_TD:g}.csv"

comparison_df.to_csv(comparison_csv, index=False)
reset_comparison_df.to_csv(reset_csv, index=False)
example_df.to_csv(OUTPUT_DIR / "engineering_example.csv", index=False)

print("CSV export complete.")


## Validation checks

In [ ]:
# Validate each operating-time equation against an explicit calculation.
M_test = 5.0
TD_test = 1.25

expected_operating = {
    "U1": TD_test * (0.0226 + 0.0104 / (M_test**0.02 - 1.0)),
    "U2": TD_test * (0.180 + 5.95 / (M_test**2 - 1.0)),
    "U3": TD_test * (0.0963 + 3.88 / (M_test**2 - 1.0)),
    "U4": TD_test * (0.0352 + 5.67 / (M_test**2 - 1.0)),
    "U5": TD_test * (0.00262 + 0.00342 / (M_test**0.02 - 1.0)),
}

for code, expected in expected_operating.items():
    calculated = relay_operating_time(code, M_test, TD_test)
    assert np.isclose(calculated, expected, rtol=1e-12, atol=1e-12), code

# Validate each reset-time equation against an explicit calculation.
M_reset_test = 0.8
expected_reset = {
    "U1": TD_test * (1.08 / (1.0 - M_reset_test**2)),
    "U2": TD_test * (5.95 / (1.0 - M_reset_test**2)),
    "U3": TD_test * (3.88 / (1.0 - M_reset_test**2)),
    "U4": TD_test * (5.67 / (1.0 - M_reset_test**2)),
    "U5": TD_test * (0.323 / (1.0 - M_reset_test**2)),
}

for code, expected in expected_reset.items():
    calculated = relay_reset_time(code, M_reset_test, TD_test)
    assert np.isclose(calculated, expected, rtol=1e-12, atol=1e-12), code

# Check invalid regions.
for code in CURVE_CODES:
    assert np.isnan(relay_operating_time(code, 1.0, 1.0))
    assert np.isnan(relay_operating_time(code, 0.9, 1.0))
    assert np.isnan(relay_reset_time(code, 1.0, 1.0))
    assert np.isnan(relay_reset_time(code, 1.1, 1.0))

# Check that generated operating data contains only positive finite values.
for code in CURVE_CODES:
    for td in TIME_DIALS:
        m, t = generate_operating_curve(code, td)
        finite_t = t[np.isfinite(t)]
        assert finite_t.size > 0
        assert np.all(finite_t > 0)

print("All validation checks passed.")


## Output quality checks

In [ ]:
required_png = []
required_svg = []

for code in CURVE_CODES:
    stem = f"{code}_{CURVES[code]['slug']}"
    required_png.append(OUTPUT_DIR / f"{stem}.png")
    required_svg.append(OUTPUT_DIR / f"{stem}.svg")

required_png.extend(
    [
        OUTPUT_DIR / f"comparison_TD{COMPARISON_TD:g}.png",
        OUTPUT_DIR / f"reset_comparison_TD{RESET_COMPARISON_TD:g}.png",
    ]
)
required_svg.extend(
    [
        OUTPUT_DIR / f"comparison_TD{COMPARISON_TD:g}.svg",
        OUTPUT_DIR / f"reset_comparison_TD{RESET_COMPARISON_TD:g}.svg",
    ]
)

for file_path in required_png + required_svg:
    assert file_path.exists(), f"Missing output: {file_path}"

for code in CURVE_CODES:
    df = curve_dataframes[code]
    assert len([column for column in df.columns if column.startswith("TD_")]) == len(TIME_DIALS)
    numeric_values = df.drop(columns="M").to_numpy()
    assert np.all(np.isfinite(numeric_values))
    assert np.all(numeric_values > 0)

assert set(comparison_df.columns[1:]) == set(CURVE_CODES)

print("All expected plots and data structures were generated successfully.")


## Create ZIP package

In [ ]:
zip_base = OUTPUT_DIR.parent / "relay_curve_results"
zip_path = Path(
    shutil.make_archive(
        str(zip_base),
        "zip",
        root_dir=OUTPUT_DIR,
    )
)

print(f"ZIP package created: {zip_path}")


## Optional Google Colab download

The following cell downloads the ZIP package automatically only when the notebook is running inside Google Colab. In a standard Jupyter environment it simply prints the package location.


In [ ]:
try:
    from google.colab import files

    files.download(str(zip_path))
except ImportError:
    print("Google Colab is not detected.")
    print(f"Download the ZIP package manually from: {zip_path}")


## Usage notes

- **\(M\)** is the current divided by pickup current.
- **Time Dial** scales the operating characteristic vertically.
- Operating equations are evaluated only for **\(M>1\)**.
- Reset equations are evaluated only for **\(0<M<1\)**.
- The point **\(M=1\)** is singular and is intentionally excluded.
- The five inverse families differ in how strongly operating time decreases as current increases.
- All numerical curve points are generated from the equations; no curve data are hard-coded.

Because the calculation layer is separated from the plotting layer, the notebook can be extended later with fuse curves, breaker curves, transformer damage curves, motor starting envelopes, coordination margins, or automated relay-setting studies.
